<a href="https://colab.research.google.com/github/pranavchauhann/sentiment-finetuning/blob/main/sentiment_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.cuda.is_available())

True


In [2]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 4.0.0


In [4]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
dataset["train"][0]


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

### Checking a Positive Review

Now we inspect another training example to see a positive sentiment review.

In this dataset:
- `0` = Negative
- `1` = Positive

In [8]:
dataset["train"][20000]

{'text': "After reading some quite negative views for this movie, I was not sure whether I should fork out some money to rent it. However, it was a pleasant surprise. I haven't seen the original movie, but if its better than this, I'd be in heaven.<br /><br />Tom Cruise gives a strong performance as the seemingly unstable David, convincing me that he is more than a smile on legs (for only the third time in his career- the other examples were Magnolia and Born on the Fourth of July). Penelope Cruz is slightly lightweight but fills the demands for her role, as does Diaz. The only disappointment is the slightly bland Kurt Russell. In the movie, however, it is not the acting that really impresses- its the filmmaking.<br /><br />Cameron Crowe excels in the director's role, providing himself with a welcome change of pace from his usual schtick. The increasing insanity of the movie is perfectly executed by Crowe (the brief sequence where Cruise walks through an empty Time Square is incredibly

### Checking Label Distribution

Before training, we should check how many positive and negative examples are present in the training dataset.

A balanced dataset usually helps the model learn both classes fairly.

In [9]:
from collections import Counter

label_counts = Counter(dataset["train"]["label"])
label_counts

Counter({0: 12500, 1: 12500})

### Viewing Text and Label Separately

Each training example contains two main fields:

- `text` → the movie review
- `label` → the sentiment class

We can access them separately to understand the dataset structure more clearly.

In [10]:
example = dataset["train"][0]

print("Review:")
print(example["text"])

print("\nLabel:")
print(example["label"])

Review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betw

### Creating a Validation Set

The IMDb dataset already has training and test sets, but it does not provide a separate validation set.

We will split the training data into:

- 90% Training
- 10% Validation

The model will learn from the training set, while the validation set will help us monitor performance during fine-tuning.

In [11]:
split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

split_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
})

### Renaming the Validation Split

The `train_test_split()` function names the second split as `test` by default.

In our project, this split will be used for validation, so we will store it with a clearer name.

In [12]:
train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Train: 22500
Validation: 2500
Test: 25000


### Loading the DistilBERT Tokenizer

Transformer models cannot directly understand raw text.

A tokenizer converts text into numerical tokens that the model can process.

We will use the tokenizer that belongs to the pretrained DistilBERT model.

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Tokenizing a Single Sentence

Let's pass a simple sentence through the DistilBERT tokenizer.

The tokenizer will convert the text into numerical IDs that DistilBERT can understand.

In [14]:
sentence = "I love this movie"

encoded = tokenizer(sentence)

encoded

{'input_ids': [101, 1045, 2293, 2023, 3185, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

### Viewing the Tokens

We can convert the numerical token IDs back into readable tokens.

This helps us understand how the tokenizer split the sentence.

In [15]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

tokens

['[CLS]', 'i', 'love', 'this', 'movie', '[SEP]']

### Understanding Subword Tokenization

DistilBERT does not always treat one word as one token.

If a word is uncommon or complex, the tokenizer may split it into smaller subword pieces.

In [16]:
word = "unbelievable"

tokens = tokenizer.tokenize(word)

tokens

['unbelievable']

### Seeing Subword Tokenization in Action

Some uncommon words are not stored as complete tokens in the tokenizer vocabulary.

In that case, the tokenizer splits them into smaller subword pieces.

In [17]:
word = "unbelievableness"

tokens = tokenizer.tokenize(word)

tokens

['unbelievable', '##ness']

### Creating a Tokenization Function

We will create a small function that takes a batch of movie reviews and converts them into tokenized inputs for DistilBERT.

We will use truncation so that very long reviews do not exceed the model's maximum input length.

In [18]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True
    )

### Testing the Tokenization Function

Before applying the function to the entire dataset, we will test it on a single training example.

This helps us verify that the function is working correctly.

In [19]:
sample = train_dataset[0]

tokenized_sample = tokenize_function(sample)

tokenized_sample

{'input_ids': [101, 2007, 2122, 2111, 6904, 6834, 2061, 2116, 7171, 1010, 2478, 2214, 8333, 1010, 1998, 3806, 7741, 4176, 2000, 2131, 2068, 2041, 1010, 2025, 2000, 5254, 2008, 2070, 1997, 1996, 5019, 2020, 6361, 2006, 1037, 2580, 2275, 2007, 5889, 1010, 2054, 1005, 1055, 2000, 2903, 1029, 2214, 2143, 1997, 3032, 2003, 3835, 1010, 2021, 1996, 4111, 6905, 1998, 16627, 1997, 12493, 2003, 9145, 2000, 3422, 1999, 2122, 3152, 1012, 1045, 2113, 1010, 14398, 2003, 7929, 1999, 2122, 2214, 3152, 1010, 2021, 2045, 2003, 2062, 2000, 2008, 2000, 2191, 2023, 3232, 4558, 21553, 1012, 6791, 2004, 13109, 10136, 1010, 2027, 2196, 5520, 2037, 9738, 1010, 3235, 3779, 2001, 2019, 4654, 1011, 12436, 12672, 26548, 2937, 1010, 2109, 2814, 2066, 2990, 2414, 2005, 3361, 5114, 2096, 10551, 2075, 2068, 1997, 25335, 1010, 16039, 2010, 2564, 1005, 1055, 6835, 6245, 1010, 2478, 2014, 2004, 1037, 10140, 17678, 1010, 2035, 2023, 3084, 2122, 3152, 24257, 1012, 2027, 2020, 2011, 2053, 2965, 1996, 2034, 2000, 3604, 2000,

### Understanding Padding

When multiple sentences are processed together in a batch, they may have different lengths.

Padding adds extra `[PAD]` tokens to shorter sentences so that all inputs in the batch have the same length.

The attention mask uses:
- `1` for real tokens
- `0` for padding tokens

In [20]:
sentences = [
    "I love this movie",
    "This movie was absolutely amazing and I really enjoyed watching it"
]

encoded_batch = tokenizer(
    sentences,
    padding=True
)

encoded_batch

{'input_ids': [[101, 1045, 2293, 2023, 3185, 102, 0, 0, 0, 0, 0, 0, 0], [101, 2023, 3185, 2001, 7078, 6429, 1998, 1045, 2428, 5632, 3666, 2009, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

### Tokenizing the Full Dataset

Now that our tokenization function is working correctly, we will apply it to the training, validation, and test datasets.

We use batched processing so multiple examples are tokenized efficiently.

In [21]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

### Inspecting the Tokenized Dataset

After tokenization, new model-ready fields are added to each example.

Let's inspect the dataset structure to see which columns are now available.

In [22]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 22500
})

### Inspecting One Tokenized Example

Let's inspect one tokenized training example.

This will help us see the original review, its label, and the numerical inputs created for DistilBERT.

In [25]:
sample = tokenized_train[0]

print("Label:", sample["label"])
print("Input IDs:", sample["input_ids"])
print("Attention Mask:", sample["attention_mask"])

Label: 0
Input IDs: [101, 2007, 2122, 2111, 6904, 6834, 2061, 2116, 7171, 1010, 2478, 2214, 8333, 1010, 1998, 3806, 7741, 4176, 2000, 2131, 2068, 2041, 1010, 2025, 2000, 5254, 2008, 2070, 1997, 1996, 5019, 2020, 6361, 2006, 1037, 2580, 2275, 2007, 5889, 1010, 2054, 1005, 1055, 2000, 2903, 1029, 2214, 2143, 1997, 3032, 2003, 3835, 1010, 2021, 1996, 4111, 6905, 1998, 16627, 1997, 12493, 2003, 9145, 2000, 3422, 1999, 2122, 3152, 1012, 1045, 2113, 1010, 14398, 2003, 7929, 1999, 2122, 2214, 3152, 1010, 2021, 2045, 2003, 2062, 2000, 2008, 2000, 2191, 2023, 3232, 4558, 21553, 1012, 6791, 2004, 13109, 10136, 1010, 2027, 2196, 5520, 2037, 9738, 1010, 3235, 3779, 2001, 2019, 4654, 1011, 12436, 12672, 26548, 2937, 1010, 2109, 2814, 2066, 2990, 2414, 2005, 3361, 5114, 2096, 10551, 2075, 2068, 1997, 25335, 1010, 16039, 2010, 2564, 1005, 1055, 6835, 6245, 1010, 2478, 2014, 2004, 1037, 10140, 17678, 1010, 2035, 2023, 3084, 2122, 3152, 24257, 1012, 2027, 2020, 2011, 2053, 2965, 1996, 2034, 2000, 3604,

### Loading DistilBERT for Sentiment Classification

Now we load the pretrained DistilBERT model.

Since our task has two classes:
- 0 = Negative
- 1 = Positive

we configure the model with two output labels.

In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Inspecting the Model Size

DistilBERT contains millions of trainable parameters.

These parameters are the weights that can be adjusted during fine-tuning.

In [27]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 66955010
Trainable parameters: 66955010


### Testing the Model Before Fine-Tuning

Before training, we will give the model a simple movie review.

The classification head is not yet trained for our sentiment task, so the prediction may be incorrect or uncertain.

Later, we will compare this with the prediction after fine-tuning.

In [28]:
import torch

text = "I absolutely loved this movie. It was fantastic!"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

outputs.logits

tensor([[-0.2097, -0.0705]])

### Converting Logits to Probabilities

The model outputs raw scores called logits.

We can use Softmax to convert these logits into probabilities for each class.

In [29]:
probabilities = torch.softmax(outputs.logits, dim=1)

probabilities

tensor([[0.4652, 0.5348]])

### Getting the Predicted Class

The model assigns a probability to each sentiment class.

We select the class with the highest probability as the final prediction.

In [30]:
predicted_class = torch.argmax(probabilities, dim=1).item()

print("Predicted class:", predicted_class)
print("Predicted sentiment:", "Positive" if predicted_class == 1 else "Negative")

Predicted class: 1
Predicted sentiment: Positive


### Defining Training Arguments

Training arguments control how the fine-tuning process will run.

We will define:
- number of epochs
- batch size
- learning rate
- logging and evaluation behavior

In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

### Setting Up Dynamic Padding

Movie reviews have different lengths.

Instead of padding every review to a fixed maximum length, we will pad dynamically within each batch.

This saves memory and makes training more efficient.

In [32]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Defining the Accuracy Metric

During validation, we want to measure how many sentiment predictions are correct.

Accuracy is calculated as:

Correct Predictions / Total Predictions

In [33]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

### Setting Up the Trainer

The Trainer connects our model, datasets, training settings, padding logic, and evaluation metric.

It will manage the fine-tuning loop for us.

In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

### Starting Fine-Tuning

Now we will start training the model on the IMDb sentiment dataset.

During training, the model will:

1. Process a batch of reviews
2. Make predictions
3. Calculate loss
4. Backpropagate the error
5. Update the weights
6. Repeat for all batches and epochs

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 